# Descriptive Statistics — Healthy Purchase Cycle

**Purpose:** Produce journal-ready Table 1 (Panels A–D) and a one-slide descriptive figure using the *same eligible households and trips* as the annual-state HMM. This script follows the uploaded **V30** input audit (minimum 3 observed years, complete finite model columns, valid outcomes). The descriptive statistics report **all six V30 estimation categories, including milk**.

**Inputs:** `healthy_purchase_data/processed/hmm_input/hmm_estimation_panel.parquet` (restricted; never committed to GitHub).

**Outputs** under `healthy_purchase_cycle/descriptive_statistics/`:
- `tables/table1_descriptive_statistics.csv`
- `tables/deck_sample_summary.csv`
- `tables/deck_category_summary.csv`
- `figures/deck_descriptive_figure.png`

**Run:** Install `duckdb`, `pandas`, `numpy`, and `matplotlib` in your local Python environment, then **Run All**. Only aggregates (no household or trip identifiers) are exported. The notebook does not run HMM estimation.

**Important:** This version includes all six V30 categories in the sample audit, journal table, and deck figure. If the actual submission model uses different filtering rules, verify eligibility against its estimation notebook before submission.
**Overwrite behavior:** Each Run All replaces the existing three CSV files and PNG figure at the exact paths above. No version-suffixed output files are created. Close any files locked by another application before running.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import duckdb

DATA_ROOT = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data")
REPO_ROOT = Path(r"C:\Users\rwen13\Documents\healthy_purchase_cycle")
INPUT_PATH = DATA_ROOT / "processed" / "hmm_input" / "hmm_estimation_panel.parquet"
OUTPUT_ROOT = REPO_ROOT / "descriptive_statistics"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
TEMP_DIR = DATA_ROOT / "temp" / "descriptive_statistics_duckdb"

MIN_OBSERVED_YEARS_PER_HOUSEHOLD = 3  # V30 default: at least 3 distinct observed panel years.
AUDIT_ALL_V30_CATEGORIES = True       # Require all six V30 categories in the input validation.
DUCKDB_MEMORY_LIMIT = "4GB"
DUCKDB_THREADS = 4

CATEGORIES = ["fv", "beverage", "snack", "cereal", "yogurt", "milk"]
DISPLAY_NAMES = {
    "fv": "Fruits & Vegetables", "beverage": "Beverages",
    "snack": "Snacks", "cereal": "Breakfast Cereal", "yogurt": "Yogurt",
    "milk": "Milk",
}
HEALTHY_LABELS = {
    "fv": "Fresh FV", "beverage": "Healthier drink", "snack": "Healthy snack",
    "cereal": "Low-sugar cereal", "yogurt": "Fat-free yogurt",
    "milk": "Fully skimmed milk",
}
CHOICE_CODES = {c: (1, 2, 3) if c == "cereal" else (1, 2) for c in CATEGORIES}
EVENTS = {
    "event_new_child": "New child", "getting_married": "Marriage",
    "getting_divorce_or_separation": "Divorce or separation", "moving": "Moving",
    "getting_job": "Getting a job", "retirement": "Retirement",
}
DEMOGRAPHICS = ["age", "family_size_numeric", "income10k_imputed", "K_1"]

# V30's complete-data audit includes these six categories and their spending columns.
V30_SPENDING = {
    "fv": ["fresh_spending", "non_fresh_spending"],
    "milk": ["skimmed_milk_spending", "low_fat_milk_spending", "other_milk_spending"],
    "yogurt": ["skimmed_yogurt_spending", "low_fat_yogurt_spending", "other_yogurt_spending"],
    "cereal": ["low_sugar_cereal_spending", "medium_sugar_cereal_spending", "high_sugar_cereal_spending"],
    "snack": ["healthy_snack_spending", "unhealthy_snack_spending"],
    "beverage": ["healthy_beverage_spending", "unhealthy_beverage_spending"],
}

for directory in [TABLE_DIR, FIGURE_DIR, TEMP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"HMM estimation input not found: {INPUT_PATH}")

print(f"Input : {INPUT_PATH}")
print(f"Output: {OUTPUT_ROOT}")

## 1. Validate the estimation input and select eligible households

- Use `hhid` and `panel_year` as household-year keys; all sample counts refer to **retained** estimation trips.
- Reject invalid data rather than quietly removing incomplete rows, matching V30.
- Preserve the original household inclusion threshold: at least three **distinct observed** years.
- Store the DuckDB temporary workspace in the restricted local data directory, not GitHub.
- The output table and sample audit both cover all six V30 categories; the input audit matches V30's original six-category validation.

In [ ]:
def quoted(col):
    """Safely quote a SQL column identifier."""
    return '"' + col.replace('"', '""') + '"'


def setup_connection(path):
    """Create a disk-backed DuckDB view over the restricted Parquet file."""
    con = duckdb.connect()
    con.execute(f"SET threads={DUCKDB_THREADS}")
    con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
    con.execute("SET temp_directory='" + str(TEMP_DIR).replace("'", "''") + "'")
    con.from_parquet(str(path)).create_view("raw")
    return con


con = setup_connection(INPUT_PATH)
audit_categories = list(V30_SPENDING) if AUDIT_ALL_V30_CATEGORIES else CATEGORIES
required = ["hhid", "panel_year", "trip_id", "purchase_date", "food_spend",
            "initial_fresh_fv_share_2007", *DEMOGRAPHICS, *EVENTS,
            *[f"{c}_y" for c in audit_categories],
            *[col for c in audit_categories for col in V30_SPENDING[c]]]
required = list(dict.fromkeys(required))
available = {row[0] for row in con.execute("DESCRIBE raw").fetchall()}
missing = sorted(set(required) - available)
if missing:
    raise ValueError(f"Missing columns required for V30-compatible validation: {missing}")

numeric = [c for c in required if c not in {"hhid", "trip_id", "purchase_date"}]
invalid_checks = [f"{quoted(c)} IS NULL" for c in required]
invalid_checks += [f"NOT isfinite(TRY_CAST({quoted(c)} AS DOUBLE))" for c in numeric]
invalid_checks += ["panel_year NOT BETWEEN 2008 AND 2017", "panel_year != floor(panel_year)",
                   "TRY_CAST(purchase_date AS DATE) IS NULL"]
invalid_checks += [f"{quoted(c)} NOT IN (0,1)" for c in ["K_1", *EVENTS]]
for c in audit_categories:
    values = "0,1,2,3" if c == "cereal" else "0,1,2"
    invalid_checks.append(f"{quoted(c + '_y')} NOT IN ({values})")
invalid_checks += ["family_size_numeric <= 0", "age <= 0", "age > 120",
                   "income10k_imputed < 0"]

input_n, invalid_n = con.execute(
    "SELECT COUNT(*), COUNT(*) FILTER (WHERE " + " OR ".join(invalid_checks) + ") FROM raw"
).fetchone()
if input_n == 0 or invalid_n:
    raise ValueError(f"Input has {input_n:,} trips and {invalid_n:,} invalid rows. Repair upstream; no silent drops.")

low_prior, high_prior = con.execute(
    "SELECT MIN(initial_fresh_fv_share_2007), MAX(initial_fresh_fv_share_2007) FROM raw"
).fetchone()
if not (0 < low_prior < 1) or abs(high_prior - low_prior) > 1e-12:
    raise ValueError("V30 initial fresh-FV prior must be one constant strictly between zero and one.")

if con.execute("""SELECT 1 FROM raw GROUP BY hhid,panel_year,trip_id
                  HAVING COUNT(*) > 1 LIMIT 1""").fetchone():
    raise ValueError("Duplicate household-year-trip records; V30 requires unique trip keys.")

# V30 assumes constant demographic and event values within each household-year.
controls = [*DEMOGRAPHICS, *EVENTS, "initial_fresh_fv_share_2007"]
within_year_changes = " OR ".join(f"MIN({quoted(c)}) != MAX({quoted(c)})" for c in controls)
if con.execute("SELECT 1 FROM raw GROUP BY hhid,panel_year HAVING " + within_year_changes + " LIMIT 1").fetchone():
    raise ValueError("Demographics, events, or the initial prior vary within a household-year.")

con.execute(f"""CREATE TEMP TABLE eligible_households AS
    SELECT hhid FROM raw GROUP BY hhid
    HAVING COUNT(DISTINCT panel_year) >= {MIN_OBSERVED_YEARS_PER_HOUSEHOLD}""")
con.execute("CREATE TEMP VIEW trips AS SELECT r.* FROM raw r SEMI JOIN eligible_households e USING (hhid)")
retained_n = con.execute("SELECT COUNT(*) FROM trips").fetchone()[0]
if retained_n == 0:
    raise ValueError("No observations satisfy the household-year eligibility threshold.")
print(f"Input trips: {input_n:,}; retained trips: {retained_n:,}; excluded trips: {input_n-retained_n:,}")

## 2. Household-year summary and sample characteristics

Demographic means/SDs are **equally weighted across household-years**, not trip weighted. Years observed per household and trips per household are **equally weighted across households**. Life-event incidence uses household-years **after the first observed year** because only those observations enter the V30 transition likelihood. This is a descriptive frequency, not a causal event-study rate.

In [ ]:
annual_fields = ["CAST(hhid AS VARCHAR) AS hhid", "CAST(panel_year AS INTEGER) AS panel_year",
                 "COUNT(*) AS n_trips"]
annual_fields += [f"FIRST({quoted(c)}) AS {quoted(c)}" for c in [*DEMOGRAPHICS, *EVENTS]]

# Materialize ONLY household-year summaries (much smaller than the trip-level input).
annual_sql = "SELECT " + ", ".join(annual_fields) + " FROM trips GROUP BY hhid, panel_year"
hy = con.execute(annual_sql).df().sort_values(["hhid", "panel_year"]).reset_index(drop=True)
hy["is_transition_year"] = hy.groupby("hhid", sort=False).cumcount().gt(0)
by_household = hy.groupby("hhid", sort=False).agg(
    observed_years=("panel_year", "size"), total_trips=("n_trips", "sum")
)
num_households = len(by_household)
num_hy = len(hy)
num_trips = int(hy.n_trips.sum())
assert num_trips == retained_n
assert int(by_household.observed_years.min()) >= MIN_OBSERVED_YEARS_PER_HOUSEHOLD
transition_hy = hy.loc[hy.is_transition_year]

sample_rows = [
    ("Sample period", "2008–2017", "Calendar years in the final estimation input"),
    ("Number of households", num_households, "Unique retained households"),
    ("Household-year observations", num_hy, "Unique retained household-years"),
    ("Shopping trips", num_trips, "Retained estimation trips"),
    ("Years observed per household", by_household.observed_years.mean(), "Household-weighted mean"),
    ("Shopping trips per household-year", hy.n_trips.mean(), "Household-year-weighted mean"),
]
summary_sample = pd.DataFrame(sample_rows, columns=["variable", "value", "definition"])
# Overwrite the previous aggregate CSV on every run.
summary_sample.to_csv(TABLE_DIR / "deck_sample_summary.csv", index=False)

# Mean and sample SD of annual covariates, not weights by grocery-shopping frequency.
demo_order = [
    ("Age of household head (years)", "age"),
    ("Household size", "family_size_numeric"),
    ("Household income ($10,000)", "income10k_imputed"),
    ("Children under 18 (indicator)", "K_1"),
]
demo_rows = [(label, float(hy[col].mean()), float(hy[col].std(ddof=1))) for label, col in demo_order]
event_rows = [(label, float(100 * transition_hy[col].mean())) for col, label in EVENTS.items()]
print(f"Households: {num_households:,} | household-years: {num_hy:,} | trips: {num_trips:,}")
print(f"Transition-eligible household-years: {len(transition_hy):,}")

## 3. Conditional category choices from retained shopping trips

- **Purchase incidence:** `COUNT(y > 0) / COUNT(all retained trips)`.
- **Healthy / intermediate / unhealthy shares:** `COUNT(y == code) / COUNT(y > 0)`.
- For cereal only, code 2 = Medium sugar and code 3 = High sugar. For binary categories, code 2 = less-healthy alternative. A missing category purchase is **not** an unhealthy choice.
- Outcomes follow the existing *healthy-first* priority rule, so these are **trip-level classifications**, not expenditure shares.

In [ ]:
category_expressions = ["COUNT(*) AS all_trips"]
for cat in CATEGORIES:
    col = quoted(cat + "_y")
    category_expressions.append(f"COUNT(*) FILTER (WHERE {col} > 0) AS {cat}_purchase")
    for choice in CHOICE_CODES[cat]:
        category_expressions.append(f"COUNT(*) FILTER (WHERE {col} = {choice}) AS {cat}_choice{choice}")
category_counts = con.execute("SELECT " + ", ".join(category_expressions) + " FROM trips").df().iloc[0]
if int(category_counts["all_trips"]) != num_trips:
    raise AssertionError("Trip denominator does not match household-year aggregation")

category_rows = []
for cat in CATEGORIES:
    n_purchase = int(category_counts[f"{cat}_purchase"])
    n_healthy = int(category_counts[f"{cat}_choice1"])
    n_intermediate = int(category_counts[f"{cat}_choice2"]) if cat == "cereal" else None
    n_unhealthy = int(category_counts[f"{cat}_choice3"] if cat == "cereal" else category_counts[f"{cat}_choice2"])
    counts = [n_healthy, n_unhealthy] if cat != "cereal" else [n_healthy, n_intermediate, n_unhealthy]
    assert sum(counts) == n_purchase, f"Category outcomes not exhaustive: {cat}"
    if n_purchase == 0:
        raise ValueError(f"No category purchases in estimation sample: {cat}")
    category_rows.append({
        "category": DISPLAY_NAMES[cat], "category_key": cat,
        "healthy_option": HEALTHY_LABELS[cat],
        "purchase_incidence_pct": 100 * n_purchase / num_trips,
        "healthy_pct": 100 * n_healthy / n_purchase,
        "intermediate_pct": (100 * n_intermediate / n_purchase) if cat == "cereal" else np.nan,
        "unhealthy_pct": 100 * n_unhealthy / n_purchase,
        "category_purchase_trips": n_purchase, "healthy_choice_trips": n_healthy,
        "intermediate_choice_trips": n_intermediate,
        "unhealthy_choice_trips": n_unhealthy,
        "all_estimation_trips": num_trips,
    })
category_df = pd.DataFrame(category_rows)
# Overwrite the previous aggregate CSV on every run.
category_df.to_csv(TABLE_DIR / "deck_category_summary.csv", index=False, float_format="%.6f")
print(category_df[["category", "purchase_incidence_pct", "healthy_pct", "intermediate_pct", "unhealthy_pct"]].round(2).to_string(index=False))

## 4. Assemble journal-ready Table 1 (one CSV, four panels)

The long-format CSV keeps panel names and statistic columns explicit. `value` contains the formatted display statistic; `mean`, `sd`, and the percentage columns are available separately for editing the manuscript table. Blank **intermediate** cells in binary categories are structurally inapplicable, not zero.

In [ ]:
table_rows = []

def add_row(panel, characteristic, value="", mean=np.nan, sd=np.nan,
            incidence=np.nan, healthy=np.nan, intermediate=np.nan, unhealthy=np.nan):
    table_rows.append({
        "panel": panel, "characteristic": characteristic, "value": value,
        "mean": mean, "sd": sd, "purchase_incidence_pct": incidence,
        "healthy_pct": healthy, "intermediate_pct": intermediate,
        "unhealthy_pct": unhealthy,
    })

for variable, value, _definition in sample_rows:
    text_value = value if isinstance(value, str) else (f"{value:,.0f}" if isinstance(value, (int, np.integer)) else f"{value:,.2f}")
    add_row("Panel A. Sample Characteristics", variable, text_value)

for variable, mean, sd in demo_rows:
    add_row("Panel B. Household Demographics", variable,
            f"{mean:.2f} ({sd:.2f})", mean=mean, sd=sd)

for _, r in category_df.iterrows():
    add_row("Panel C. Category-Level Purchase Behavior", r["category"],
            incidence=r["purchase_incidence_pct"], healthy=r["healthy_pct"],
            intermediate=r["intermediate_pct"], unhealthy=r["unhealthy_pct"])

for label, value in event_rows:
    add_row("Panel D. Life Events", label, f"{value:.2f}%")

journal_table = pd.DataFrame(table_rows)
path_table = TABLE_DIR / "table1_descriptive_statistics.csv"
# Overwrite the previous journal table on every run.
journal_table.to_csv(path_table, index=False, float_format="%.4f")
print(f"Saved: {path_table}")
print("Notes: Panel B unit = household-year; Panel D unit = transition-eligible household-year.")

## 5. Build the deck figure

Layout based on the supplied visual reference: four left-side KPI cards and two right-side horizontal bar charts. The charts include **all six V30 categories**, including yogurt; the healthy cereal label is **Low-sugar cereal** (not Adult cereal). A 300-DPI PNG is saved for PowerPoint.

In [ ]:
from matplotlib.patches import FancyBboxPatch, Circle

PALETTE = {"fv": "#26883e", "beverage": "#1877c4", "snack": "#f18c1a",
           "cereal": "#7544af", "yogurt": "#ca4c70", "milk": "#0e9099"}

fig = plt.figure(figsize=(16, 9), facecolor="#fafafa")
fig.subplots_adjust(0, 0, 1, 1)
background = fig.add_axes([0, 0, 1, 1], zorder=0)
background.axis("off")

def rounded_panel(x, y, w, h, edge="#dfdfdf", fill="#ffffff", linewidth=1.25):
    patch = FancyBboxPatch((x, y), w, h, transform=background.transAxes,
                           boxstyle="round,pad=0.008,rounding_size=0.021",
                           linewidth=linewidth, edgecolor=edge, facecolor=fill)
    background.add_patch(patch)

def kpi_box(bottom, color, number, label, symbol):
    rounded_panel(0.025, bottom, 0.315, 0.205, edge=color, fill="#ffffff", linewidth=1.5)
    background.add_patch(Circle((0.082, bottom + 0.102), 0.055,
                                transform=background.transAxes, facecolor="#f7f9fb",
                                edgecolor=color, linewidth=1.5))
    background.text(0.082, bottom + 0.101, symbol, ha="center", va="center",
                    fontsize=28, color=color, weight="bold", transform=background.transAxes)
    background.text(0.145, bottom + 0.127, number, ha="left", va="center",
                    fontsize=38, weight="bold", color=color, transform=background.transAxes)
    background.text(0.145, bottom + 0.047, label, ha="left", va="center",
                    fontsize=16, color="#202020", transform=background.transAxes)

kpi_box(0.757, "#168546", f"{num_households:,}", "Households", "H")
kpi_box(0.521, "#1877c4", f"{num_hy:,}", "Household-years", "Y")
kpi_box(0.285, "#7544af", f"{num_trips/1_000_000:.2f}M", "Shopping trips", "T")
kpi_box(0.049, "#f09a12", f"{num_trips/num_households:,.1f}", "Mean trips per household", "#")

rounded_panel(0.373, 0.52, 0.607, 0.443)
rounded_panel(0.373, 0.049, 0.607, 0.443)

positions = [(0.47, "Category purchase incidence", "purchase_incidence_pct", "category"),
             (0.091, "Healthy-option share within category trips", "healthy_pct", "healthy_option")]
# Give the six categories sufficient room in both right-hand panels.
for top, title, metric, label_col in positions:
    axes = fig.add_axes([0.57, top + 0.047, 0.355, 0.285], zorder=2)
    vals = category_df[metric].to_numpy(float)
    labels = category_df[label_col].tolist()
    keys = category_df["category_key"].tolist()
    y = np.arange(len(keys))
    bars = axes.barh(y, vals, color=[PALETTE[k] for k in keys], height=0.57)
    axes.set_yticks(y, labels=labels, fontsize=10)
    axes.invert_yaxis()
    axes.set_xlim(0, max(100, np.max(vals) + 13))
    axes.set_xticks([0, 20, 40, 60, 80, 100])
    axes.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
    axes.grid(axis="x", linestyle="--", color="#dddddd", linewidth=0.8)
    axes.set_axisbelow(True)
    for spine in ["top", "right", "left"]:
        axes.spines[spine].set_visible(False)
    axes.spines["bottom"].set_color("#999999")
    axes.tick_params(axis="both", length=0, pad=7)
    for bar, val in zip(bars, vals):
        axes.text(val + 1.2, bar.get_y() + bar.get_height()/2,
                  f"{val:.1f}%", va="center", ha="left", fontsize=10, weight="bold")
    background.text(0.397, top + 0.355, title, fontsize=16, weight="medium",
                    color="#222222", transform=background.transAxes)

background.text(0.375, 0.012, "2008–2017  |  Six-category V30 HMM sample  |  Conditional shares use category-purchase trips",
                fontsize=10, color="#777777", transform=background.transAxes)
figure_path = FIGURE_DIR / "deck_descriptive_figure.png"
# Overwrite the previous deck figure on every run.
fig.savefig(figure_path, dpi=300, facecolor=fig.get_facecolor(), bbox_inches="tight", pad_inches=0.18)
plt.show()
print(f"Saved: {figure_path}")

## 6. Final audit and interpretation notes

**Denominators**

1. Panel A: retained eligible households and trips. `Years observed per household` is household weighted; `trips per household-year` is household-year weighted.
2. Panel B: means and sample SDs across observed household-years. `K_1` is a 0/1 indicator; its mean equals the fraction of household-years with children under 18.
3. Panel C: purchase incidence across **all retained shopping trips**; choice shares across **category-purchase trips only**. The shares use gated choice codes and do not equal category dollar shares.
4. Panel D: unweighted fraction of household-years that enter the transition model (every observed year other than a household's first), with the **current-year event variable**.
5. Cereal: a zero code can include trips without *classified* cereal purchases; consequently, the reported incidence is classified-cereal incidence, not necessarily all cereal purchases.
6. Milk is included as a sixth category: `milk_y = 1` denotes a trip with at least one fully skimmed milk purchase; `milk_y = 2` denotes another milk purchase without fully skimmed milk. This is a gated trip-level choice, not the spending share on skimmed milk. Confirm that submission-model filtering matches V30 before describing aggregates as a final-model sample match.

**Data governance:** The notebook uses the separate local restricted data directory for Parquet and temporary storage, and exports only aggregate CSV/PNG outputs to the requested GitHub repository folder. Review outputs for your data-sharing agreement before committing them.